In [1]:
import os, subprocess

!pip install -q opencv-python-headless python_speech_features scipy tqdm numpy
!pip install -q "scenedetect[opencv]>=0.6.3"

!rm -rf /kaggle/working/syncnet_python
!git clone https://github.com/joonson/syncnet_python.git /kaggle/working/syncnet_python

SYNCNET_DIR = "/kaggle/working/syncnet_python"
assert os.path.exists(f"{SYNCNET_DIR}/run_pipeline.py"), "Clone failed."

os.makedirs(f"{SYNCNET_DIR}/data", exist_ok=True)
os.makedirs(f"{SYNCNET_DIR}/detectors/s3fd/weights", exist_ok=True)

!wget -q https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/syncnet_v2.model \
    -O {SYNCNET_DIR}/data/syncnet_v2.model
!wget -q https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/sfd_face.pth \
    -O {SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth

for w in [f"{SYNCNET_DIR}/data/syncnet_v2.model",
          f"{SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth"]:
    sz = os.path.getsize(w) if os.path.exists(w) else 0
    print(f"{w.split('/')[-1]}: {sz/1e6:.1f} MB")
    assert sz > 1_000_000, f"{w} download failed."

!nvidia-smi --query-gpu=index,name,memory.total --format=csv
print("Setup complete.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 6.7 MB/s eta 0:00:00
Cloning into '/kaggle/working/syncnet_python'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 155 (delta 77), reused 75 (delta 67), pack-reused 55 (from 1)
Receiving objects: 100% (155/155), 106.12 KiB | 2.87 MiB/s, done.
Resolving deltas: 100% (96/96), done.
syncnet_v2.model: 54.6 MB
sfd_face.pth: 89.8 MB
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
Setup complete.


In [2]:
import os, glob, math, json, subprocess, shutil, sys, time, threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
N_CPUS = os.cpu_count()
print(f"GPUs: {N_GPUS}   CPU cores: {N_CPUS}")

SYNCNET_DIR = "/kaggle/working/syncnet_python"
WORK_DIR    = "/kaggle/working"

# ---------------- dataset root ----------------
DATA_ROOT = "/kaggle/input/datasets/mustakimbinmohsin/speech/dataset"
if not os.path.isdir(DATA_ROOT):
    # fall back to a broad search if the exact path shifts
    hits = [p for p in glob.glob("/kaggle/input/**/dataset", recursive=True)
            if os.path.isdir(os.path.join(p, "train"))]
    assert hits, "Dataset root not found under /kaggle/input"
    DATA_ROOT = hits[0]
print("DATA_ROOT:", DATA_ROOT)

VIDEO_EXTS = (".mpg", ".mp4", ".avi", ".mov", ".mkv")

# ---------------- detect directory layout ----------------
# A: <root>/<split>/video/<speaker>/*.mpg
# B: <root>/<split>/<speaker>/video/*.mpg
def _count(pattern):
    return len([f for f in glob.glob(pattern) if f.lower().endswith(VIDEO_EXTS)])

n_A = _count(os.path.join(DATA_ROOT, "train", "video", "*", "*"))
n_B = _count(os.path.join(DATA_ROOT, "train", "*", "video", "*"))
LAYOUT = "A" if n_A >= n_B else "B"
print(f"Layout A (split/video/spk): {n_A} files | Layout B (split/spk/video): {n_B} files")
print(f"--> using layout {LAYOUT}")
assert max(n_A, n_B) > 0, "No video files found in either layout."

def video_dir(split, speaker):
    """Directory holding a speaker's videos for a split."""
    return (os.path.join(DATA_ROOT, split, "video", speaker) if LAYOUT == "A"
            else os.path.join(DATA_ROOT, split, speaker, "video"))

def list_videos(split, speaker):
    d = video_dir(split, speaker)
    if not os.path.isdir(d):
        return []
    return sorted(f for f in os.listdir(d) if f.lower().endswith(VIDEO_EXTS))

# ---------------- speakers ----------------
if LAYOUT == "A":
    spk_src = os.path.join(DATA_ROOT, "train", "video")
else:
    spk_src = os.path.join(DATA_ROOT, "train")
SPEAKERS = sorted(
    [d for d in os.listdir(spk_src) if os.path.isdir(os.path.join(spk_src, d))
     and d not in ("video", "audio")],
    key=lambda s: int("".join(ch for ch in s if ch.isdigit()) or 0)
)
print("Speakers:", SPEAKERS)

for split in ["train", "test"]:
    counts = {s: len(list_videos(split, s)) for s in SPEAKERS}
    print(f"  {split}: {sum(counts.values())} videos  {counts}")

# ---------------- manifest (informational) ----------------
for mp in [os.path.join(DATA_ROOT, "manifest.csv"),
           os.path.join(os.path.dirname(DATA_ROOT), "manifest.csv")]:
    if os.path.exists(mp):
        df = pd.read_csv(mp)
        print(f"\nmanifest.csv: {len(df)} rows | columns: {list(df.columns)}")
        print(df.head(3).to_string())
        break

# ---------------- output dirs ----------------
TRAIN_PROC_DIR = f"{WORK_DIR}/proc_train"
TEST_PROC_DIR  = f"{WORK_DIR}/proc_test"
TRAIN_FEAT_DIR = f"{WORK_DIR}/features/train"
TEST_FEAT_DIR  = f"{WORK_DIR}/features/test"
CKPT_DIR       = f"{WORK_DIR}/checkpoints"
for d in [TRAIN_PROC_DIR, TEST_PROC_DIR, TRAIN_FEAT_DIR, TEST_FEAT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------------- paper hyperparameters (Sec. 4.1) ----------------
TAU      = 15            # max delay offset -> distribution length 2*tau+1
DIST_LEN = 2 * TAU + 1   # 31
FPS, SR  = 25, 16000
AR_BLOCKS, AR_HEADS, AR_DIM, DROPOUT = 2, 16, 256, 0.1
LR, WEIGHT_DECAY = 1e-3, 1e-6
BATCH_SIZE   = 16 * max(1, N_GPUS)
EPOCHS       = 40
WARMUP_STEPS = 300
SEQ_LEN      = None      # set in Cell 7 from actual feature lengths

print(f"\nDIST_LEN={DIST_LEN}  batch={BATCH_SIZE}  epochs={EPOCHS}")

GPUs: 2   CPU cores: 4
DATA_ROOT: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset
Layout A (split/video/spk): 6400 files | Layout B (split/spk/video): 0 files
--> using layout A
Speakers: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8']
  train: 6400 videos  {'s1': 800, 's2': 800, 's3': 800, 's4': 800, 's5': 800, 's6': 800, 's7': 800, 's8': 800}
  test: 1600 videos  {'s1': 200, 's2': 200, 's3': 200, 's4': 200, 's5': 200, 's6': 200, 's7': 200, 's8': 200}

manifest.csv: 8000 rows | columns: ['speaker', 'utterance_id', 'audio_path', 'video_path', 'split']
  speaker utterance_id                                                   audio_path                                                   video_path  split
0      s1       sbia1s  D:\Audio+video\gridcorpus\dataset\train\audio\s1\sbia1s.wav  D:\Audio+video\gridcorpus\dataset\train\video\s1\sbia1s.mpg  train
1      s1       pbbc7a  D:\Audio+video\gridcorpus\dataset\train\audio\s1\pbbc7a.wav  D:\Audio+video\gridcorpus\dataset\train\vide

In [3]:
preprocess_worker_code = r'''
import sys, os, subprocess, time, shutil

speaker_list = sys.argv[1].split(",")
video_root   = os.path.abspath(sys.argv[2])   # <root>/<split>/video
out_data_dir = os.path.abspath(sys.argv[3])
gpu_id       = sys.argv[4]
SYNCNET_DIR  = "/kaggle/working/syncnet_python"

os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
import torch
print(f"[GPU {gpu_id}] cuda={torch.cuda.is_available()}", flush=True)

os.makedirs(out_data_dir, exist_ok=True)

jobs = []
for spk in speaker_list:
    vdir = os.path.join(video_root, spk)
    if os.path.isdir(vdir):
        for f in sorted(os.listdir(vdir)):
            if f.lower().endswith((".mpg", ".mp4", ".avi")):
                jobs.append((spk, f))

total = len(jobs); n_ok = n_fail = 0; shown = False; t0 = time.time()
print(f"[GPU {gpu_id}] starting: {total} videos", flush=True)

for idx, (spk, vf) in enumerate(jobs, 1):
    vpath = os.path.join(video_root, spk, vf)
    ref   = f"{spk}_{os.path.splitext(vf)[0]}"
    check = os.path.join(out_data_dir, "pycrop", ref)

    if glob_has := (os.path.isdir(check) and
                    any(x.endswith(".avi") for x in os.listdir(check))):
        n_ok += 1
        continue

    r = subprocess.run(
        ["python", "run_pipeline.py",
         "--videofile", os.path.abspath(vpath),
         "--reference", ref,
         "--data_dir", out_data_dir,
         "--min_track", "25",        # GRID clips are ~75 frames; default 100 rejects all
         "--min_face_size", "40",    # 360x288 source -> faces below default 100px
         "--overwrite"],
        cwd=SYNCNET_DIR, check=False, capture_output=True, text=True)

    if os.path.isdir(check) and any(x.endswith(".avi") for x in os.listdir(check)):
        n_ok += 1
        for sub in ["pyframes", "pyavi", "pywork"]:   # free disk immediately
            shutil.rmtree(os.path.join(out_data_dir, sub, ref), ignore_errors=True)
    else:
        n_fail += 1
        if not shown:
            sys.stderr.write(f"\n[GPU {gpu_id}] FIRST FAILURE: {vpath}\n")
            sys.stderr.write(f"STDERR tail:\n{r.stderr[-500:]}\n")
            sys.stderr.flush(); shown = True
        if n_fail >= 15 and n_ok == 0:
            sys.stderr.write(f"[GPU {gpu_id}] ABORT: 15 failures, 0 successes.\n")
            sys.stderr.flush(); break

    if idx % 25 == 0:
        el = time.time() - t0
        eta = (el / idx) * (total - idx) / 60
        print(f"[GPU {gpu_id}] {idx}/{total} | ok={n_ok} fail={n_fail} "
              f"| {el/60:.1f}m | ETA ~{eta:.1f}m", flush=True)

print(f"[GPU {gpu_id}] done: {n_ok} ok, {n_fail} fail, {(time.time()-t0)/60:.1f} min",
      flush=True)
'''

extract_worker_code = r'''
import sys, os, glob, shutil, time, traceback
import numpy as np
import torch

speaker_list  = sys.argv[1].split(",")
proc_data_dir = os.path.abspath(sys.argv[2])
out_feat_dir  = os.path.abspath(sys.argv[3])
gpu_id        = sys.argv[4]
tau           = int(sys.argv[5])
SYNCNET_DIR   = "/kaggle/working/syncnet_python"

os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

class Opt:
    """evaluate() reads tmp_dir, reference, batch_size, vshift."""
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir = tmp_dir
        self.reference = reference
        self.batch_size = 20
        self.vshift = vshift

net = SyncNetInstance()
net.loadParameters(os.path.join(SYNCNET_DIR, "data", "syncnet_v2.model"))
print(f"[GPU {gpu_id}] SyncNet loaded", flush=True)

os.makedirs(out_feat_dir, exist_ok=True)
pycrop = os.path.join(proc_data_dir, "pycrop")
if not os.path.isdir(pycrop):
    print(f"[GPU {gpu_id}] no pycrop at {pycrop}", flush=True); sys.exit(0)

mine = [t for t in sorted(os.listdir(pycrop)) if t.split("_")[0] in speaker_list]
total = len(mine); n_ok = n_skip = n_fail = 0; shown = False; t0 = time.time()
print(f"[GPU {gpu_id}] starting: {total} tracks", flush=True)

for idx, track in enumerate(mine, 1):
    try:
        out_path = os.path.join(out_feat_dir, f"{track}.npy")
        if os.path.exists(out_path):
            n_skip += 1; continue

        avis = sorted(glob.glob(os.path.join(pycrop, track, "*.avi")))
        if not avis:
            n_fail += 1; continue

        tmp = os.path.join("/kaggle/working/synctmp", f"g{gpu_id}")
        os.makedirs(tmp, exist_ok=True)

        _, _, dists = net.evaluate(Opt(tmp, track, tau),
                                   videofile=os.path.abspath(avis[0]))

        # distances -> similarity -> softmax over the 2*tau+1 window = S(i,j), Eq.(1)
        probs = torch.softmax(torch.tensor(-np.stack(dists)), dim=-1).numpy()
        np.save(out_path, probs.astype(np.float32))
        shutil.rmtree(os.path.join(tmp, track), ignore_errors=True)
        n_ok += 1

    except Exception:
        n_fail += 1
        if not shown:
            sys.stderr.write(f"\n[GPU {gpu_id}] FIRST FAILURE track={track}\n")
            sys.stderr.write(traceback.format_exc()[-800:])
            sys.stderr.flush(); shown = True
        if n_fail >= 10 and n_ok == 0:
            sys.stderr.write(f"[GPU {gpu_id}] ABORT: 10 failures, 0 successes.\n")
            sys.stderr.flush(); break

    if idx % 50 == 0:
        el = time.time() - t0
        eta = (el / idx) * (total - idx) / 60
        print(f"[GPU {gpu_id}] {idx}/{total} | ok={n_ok} skip={n_skip} fail={n_fail} "
              f"| {el/60:.1f}m | ETA ~{eta:.1f}m", flush=True)

print(f"[GPU {gpu_id}] done: {n_ok} new, {n_skip} skip, {n_fail} fail, "
      f"{(time.time()-t0)/60:.1f} min", flush=True)
'''

with open("preprocess_worker.py", "w") as f: f.write(preprocess_worker_code)
with open("extract_worker.py", "w") as f: f.write(extract_worker_code)

for s in ["preprocess_worker.py", "extract_worker.py"]:
    r = subprocess.run(["python", "-m", "py_compile", s], capture_output=True, text=True)
    print(f"{s}: {'OK' if r.returncode == 0 else 'SYNTAX ERROR' + r.stderr}")

preprocess_worker.py: OK
extract_worker.py: OK


In [4]:
# Verifies ONE video end-to-end, and ONE SyncNet feature extraction,
# before committing hours of compute.
vids = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "video", "*", "*.mpg")))
assert vids, "No videos found."
print("Testing:", vids[0])

shutil.rmtree("/kaggle/working/debug_proc", ignore_errors=True)
r = subprocess.run(
    ["python", "run_pipeline.py",
     "--videofile", os.path.abspath(vids[0]),
     "--reference", "debug_test",
     "--data_dir", "/kaggle/working/debug_proc",
     "--min_track", "25", "--min_face_size", "40", "--overwrite"],
    cwd=SYNCNET_DIR, capture_output=True, text=True)

print("run_pipeline rc:", r.returncode)
crops = glob.glob("/kaggle/working/debug_proc/pycrop/debug_test/*.avi")
print("crops:", crops)
if not crops:
    print("STDERR tail:\n", r.stderr[-2000:])
assert crops, "STEP 1 FAILED -- face-track cropping produced nothing."

# --- step 2: SyncNet feature extraction on that crop ---
sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

class Opt:
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir, self.reference = tmp_dir, reference
        self.batch_size, self.vshift = 20, vshift

shutil.rmtree("/kaggle/working/debug_tmp", ignore_errors=True)
os.makedirs("/kaggle/working/debug_tmp", exist_ok=True)

net = SyncNetInstance()
net.loadParameters(f"{SYNCNET_DIR}/data/syncnet_v2.model")
offset, conf, dists = net.evaluate(
    Opt("/kaggle/working/debug_tmp", "debug_test", TAU),
    videofile=os.path.abspath(crops[0]))

d = np.stack(dists)
probs = torch.softmax(torch.tensor(-d), dim=-1).numpy()
print(f"\noffset={offset}  conf={conf:.3f}")
print(f"feature shape: {probs.shape}  (expected [T, {DIST_LEN}])")
print(f"row sums (should be ~1.0): {probs[:3].sum(axis=1)}")
assert probs.shape[1] == DIST_LEN, f"Got width {probs.shape[1]}, expected {DIST_LEN}"
print("\nDIAGNOSTIC PASSED -- proceed to Cell 5.")

Testing: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset/train/video/s1/bbaf2n.mpg
run_pipeline rc: 0
crops: ['/kaggle/working/debug_proc/pycrop/debug_test/00000.avi']


Audio (3.0041s) and video (3.0400s) lengths are different.



offset=0  conf=5.929
feature shape: (70, 31)  (expected [T, 31])
row sums (should be ~1.0): [ 1.000  1.000  1.000]

DIAGNOSTIC PASSED -- proceed to Cell 5.


In [5]:
WORKERS_PER_GPU = 2
NUM_WORKERS = WORKERS_PER_GPU * max(1, N_GPUS)

def chunk_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i*k+min(i,m):(i+1)*k+min(i+1,m)] for i in range(n)]

def launch_parallel(script, in_dir, out_dir, extra_args=None):
    """Launches all workers at once; GPUs assigned round-robin."""
    extra_args = extra_args or []
    chunks = [c for c in chunk_list(SPEAKERS, NUM_WORKERS) if c]
    procs = []
    for i, chunk in enumerate(chunks):
        gpu = str(i % max(1, N_GPUS))
        print(f"  worker {i} -> GPU {gpu}: {chunk}", flush=True)
        procs.append(subprocess.Popen(
            ["python", "-u", script, ",".join(chunk),
             os.path.abspath(in_dir), os.path.abspath(out_dir), gpu] + extra_args))
    codes = [p.wait() for p in procs]
    for i, c in enumerate(codes):
        if c != 0:
            print(f"  [warn] worker {i} exit code {c}")

def run_monitored(stage, script, in_dir, out_dir, extra_args, count_glob, target):
    """Background monitor counts real output files -- immune to stdout deadlocks."""
    stop = threading.Event()
    def mon():
        t = time.time(); n0 = len(glob.glob(count_glob)); stall = 0; last = n0
        print(f"[monitor] {stage}: from {n0}/{target}", flush=True)
        while not stop.wait(30):
            n = len(glob.glob(count_glob)); el = time.time()-t; made = n-n0
            if made > 0:
                rate = made/el
                print(f"[monitor] {stage}: {n}/{target} | {rate*60:.0f}/min "
                      f"| {el/60:.1f}m | ETA ~{(target-n)/rate/60:.1f}m", flush=True)
            else:
                print(f"[monitor] {stage}: {n}/{target} | {el/60:.1f}m | no output yet",
                      flush=True)
            stall = stall+1 if n == last else 0
            if stall >= 6:
                print("[monitor] WARNING: no progress in 3 min.", flush=True); stall = 0
            last = n
    th = threading.Thread(target=mon, daemon=True); th.start()
    t0 = time.time()
    try:
        launch_parallel(script, in_dir, out_dir, extra_args)
    finally:
        stop.set(); th.join(timeout=2)
    print(f"{stage}: {(time.time()-t0)/60:.1f} min\n", flush=True)


for split, proc_dir in [("train", TRAIN_PROC_DIR), ("test", TEST_PROC_DIR)]:
    video_root = os.path.join(DATA_ROOT, split, "video")
    target = len(glob.glob(os.path.join(video_root, "*", "*.mpg")))
    print(f"=== Preprocessing {split.upper()} ({target} videos) ===")
    run_monitored(split.upper(), "preprocess_worker.py", video_root, proc_dir, None,
                  os.path.join(proc_dir, "pycrop", "*", "*.avi"), target)

for d, name in [(TRAIN_PROC_DIR, "TRAIN"), (TEST_PROC_DIR, "TEST")]:
    n = len(glob.glob(os.path.join(d, "pycrop", "*", "*.avi")))
    print(f"{name}: {n} usable face tracks")

=== Preprocessing TRAIN (6400 videos) ===
  worker 0 -> GPU 0: ['s1', 's2']
[monitor] TRAIN: from 0/6400
  worker 1 -> GPU 1: ['s3', 's4']
  worker 2 -> GPU 0: ['s5', 's6']
  worker 3 -> GPU 1: ['s7', 's8']
[GPU 0] cuda=True
[GPU 0] starting: 1600 videos
[GPU 1] cuda=True
[GPU 1] starting: 1600 videos[GPU 1] cuda=True

[GPU 1] starting: 1600 videos
[GPU 0] cuda=True
[GPU 0] starting: 1600 videos
[monitor] TRAIN: 8/6400 | 16/min | 0.5m | ETA ~399.5m
[monitor] TRAIN: 20/6400 | 20/min | 1.0m | ETA ~319.0m
[monitor] TRAIN: 32/6400 | 21/min | 1.5m | ETA ~298.5m
[monitor] TRAIN: 44/6400 | 22/min | 2.0m | ETA ~288.9m
[monitor] TRAIN: 56/6400 | 22/min | 2.5m | ETA ~283.2m
[monitor] TRAIN: 69/6400 | 23/min | 3.0m | ETA ~275.3m
[monitor] TRAIN: 82/6400 | 23/min | 3.5m | ETA ~269.7m
[monitor] TRAIN: 94/6400 | 23/min | 4.0m | ETA ~268.4m
[GPU 0] 25/1600 | ok=25 fail=0 | 4.1m | ETA ~257.4m
[GPU 0] 25/1600 | ok=25 fail=0 | 4.1m | ETA ~259.2m
[GPU 1] 25/1600 | ok=25 fail=0 | 4.2m | ETA ~262.7m
[GPU 1


[GPU 1] FIRST FAILURE: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset/train/video/s8/bbay9p.mpg
STDERR tail:
etecting faces: 100%|██████████| 10/10 [00:00<00:00, 10.12it/s, dets=1]
2026-08-21 16:17:29,775 pyscenedetect INFO: Detecting scenes...
2026-08-21 16:17:29,803 __main__ INFO: /kaggle/working/proc_train/pyavi/s8_bbay9p/video.avi - scenes detected 1
2026-08-21 16:17:29,804 py.warnings WARNING: /kaggle/working/syncnet_python/run_pipeline.py:298: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  if shot[1].get_frames() - shot[0].get_frames() >= opt.min_track :




[monitor] TRAIN: 3225/6400 | 24/min | 131.9m | ETA ~129.8m
[monitor] TRAIN: 3237/6400 | 24/min | 132.4m | ETA ~129.4m
[monitor] TRAIN: 3249/6400 | 24/min | 132.9m | ETA ~128.9m
[monitor] TRAIN: 3261/6400 | 24/min | 133.4m | ETA ~128.4m
[monitor] TRAIN: 3273/6400 | 24/min | 133.9m | ETA ~127.9m
[GPU 1] 825/1600 | ok=824 fail=1 | 133.9m | ETA ~125.8m
[monitor] TRAIN: 3285/6400 | 24/min | 134.4m | ETA ~127.4m
[monitor] TRAIN: 3297/6400 | 24/min | 134.9m | ETA ~127.0m
[GPU 0] 825/1600 | ok=825 fail=0 | 134.9m | ETA ~126.8m
[GPU 0] 825/1600 | ok=825 fail=0 | 135.0m | ETA ~126.8m
[monitor] TRAIN: 3309/6400 | 24/min | 135.4m | ETA ~126.5m
[GPU 1] 825/1600 | ok=825 fail=0 | 135.5m | ETA ~127.3m
[monitor] TRAIN: 3319/6400 | 24/min | 135.9m | ETA ~126.2m
[monitor] TRAIN: 3333/6400 | 24/min | 136.4m | ETA ~125.5m
[monitor] TRAIN: 3345/6400 | 24/min | 136.9m | ETA ~125.0m
[monitor] TRAIN: 3358/6400 | 24/min | 137.4m | ETA ~124.5m
[GPU 1] 850/1600 | ok=845 fail=5 | 137.8m | ETA ~121.6m
[monitor] TR


[GPU 1] FIRST FAILURE: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset/test/video/s8/bbar6a.mpg
STDERR tail:
]
Detecting faces: 100%|██████████| 9/9 [00:00<00:00,  9.32it/s, dets=1]
2026-08-21 19:01:43,258 pyscenedetect INFO: Detecting scenes...
2026-08-21 19:01:43,280 __main__ INFO: /kaggle/working/proc_test/pyavi/s8_bbar6a/video.avi - scenes detected 1
2026-08-21 19:01:43,282 py.warnings WARNING: /kaggle/working/syncnet_python/run_pipeline.py:298: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  if shot[1].get_frames() - shot[0].get_frames() >= opt.min_track :




[monitor] TEST: 810/1600 | 24/min | 33.5m | ETA ~32.7m
[monitor] TEST: 822/1600 | 24/min | 34.0m | ETA ~32.2m
[monitor] TEST: 834/1600 | 24/min | 34.5m | ETA ~31.7m
[monitor] TEST: 846/1600 | 24/min | 35.0m | ETA ~31.2m
[monitor] TEST: 858/1600 | 24/min | 35.5m | ETA ~30.7m
[monitor] TEST: 870/1600 | 24/min | 36.0m | ETA ~30.2m
[monitor] TEST: 882/1600 | 24/min | 36.5m | ETA ~29.7m
[monitor] TEST: 894/1600 | 24/min | 37.0m | ETA ~29.2m
[GPU 0] 225/400 | ok=225 fail=0 | 37.0m | ETA ~28.8m
[GPU 1] 225/400 | ok=224 fail=1 | 37.1m | ETA ~28.8m
[GPU 0] 225/400 | ok=225 fail=0 | 37.1m | ETA ~28.9m
[GPU 1] 225/400 | ok=225 fail=0 | 37.3m | ETA ~29.0m
[monitor] TEST: 906/1600 | 24/min | 37.5m | ETA ~28.8m
[monitor] TEST: 918/1600 | 24/min | 38.0m | ETA ~28.3m
[monitor] TEST: 930/1600 | 24/min | 38.5m | ETA ~27.8m
[monitor] TEST: 942/1600 | 24/min | 39.0m | ETA ~27.3m
[monitor] TEST: 953/1600 | 24/min | 39.5m | ETA ~26.8m
[monitor] TEST: 966/1600 | 24/min | 40.0m | ETA ~26.3m
[monitor] TEST: 97